# I. RANDOM FOREST PIPELINE

## 1. CONFIG

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error, r2_score

DATA_PATH = "data/preprocessed_features.csv"
MODEL_DIR = "model/rf_per_target_models"
RESULTS_PATH = "results/rf_per_target_results.csv"

N_SPLITS = 5
RANDOM_STATE = 42

RF_PARAMS = dict(
    n_estimators=200,
    max_features="sqrt",
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=RANDOM_STATE
)

os.makedirs(MODEL_DIR, exist_ok=True)

## 2. LOAD DATA 

In [ ]:
df = pd.read_csv(DATA_PATH)

user_col = "pass__user_id"
output_prefixes = (
    "num__liwc_",
    "num__topic_",
    "num__genius_",
    "num__music_track_",
)

output_cols = [c for c in df.columns if c.startswith(output_prefixes)]
input_cols = [c for c in df.columns if c not in output_cols]
input_cols.remove(user_col)

X = df[input_cols].values
groups = df[user_col].values

print(f"Samples: {len(X)}")
print(f"Inputs: {len(input_cols)}")
print(f"Targets: {len(output_cols)}")

## 3. RANDOM FOREST WITH USER-BLOCKED CROSS-VALIDATION 

In [ ]:
cv = GroupKFold(n_splits=N_SPLITS)
rows = []

for target in output_cols:
    print(f"\nTraining target: {target}")

    y = df[target].values

    fold_metrics = []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), 1):
        model = RandomForestRegressor(**RF_PARAMS)
        model.fit(X[tr], y[tr])

        preds = model.predict(X[te])

        rmse = np.sqrt(mean_squared_error(y[te], preds))
        r2 = r2_score(y[te], preds)

        fold_metrics.append((rmse, r2))

    rmses, r2s = zip(*fold_metrics)

    rows.append({
        "target": target,
        "rmse_mean": np.mean(rmses),
        "rmse_std": np.std(rmses),
        "r2_mean": np.mean(r2s)
    })

    # train final model on full data
    final_model = RandomForestRegressor(**RF_PARAMS)
    final_model.fit(X, y)

    joblib.dump(
        {
            "model": final_model,
            "input_cols": input_cols
        },
        os.path.join(MODEL_DIR, f"rf_{target}.joblib")
    )

# SAVE RESULTS
results_df = pd.DataFrame(rows).sort_values("rmse_mean")
results_df.to_csv(RESULTS_PATH, index=False)

print("\n=== BEST TARGETS ===")
print(results_df.head(10))

print("\n=== WORST TARGETS ===")
print(results_df.tail(10))



# II. RANDOM FOREST INTERPRETATION

## 1. CONFIG

In [ ]:
DATA_PATH = "data/preprocessed_features.csv"
MODEL_DIR = "model/rf_per_target_models"
OUT_DIR = "results/rf_interpretation"

TOP_K = 15
N_BASE = 1000
N_PDP = 20
RANDOM_STATE = 42

os.makedirs(OUT_DIR, exist_ok=True)

## 2. LOAD DATA

In [ ]:
df = pd.read_csv(DATA_PATH)
rng = np.random.default_rng(RANDOM_STATE)

## 3. ITERATE OVER MODELS

In [ ]:

for fname in sorted(os.listdir(MODEL_DIR)):

    if not fname.endswith(".joblib"):
        continue

    target = fname.replace("rf_", "").replace(".joblib", "")
    print(f"\nInterpreting {target}")

    imp_path = os.path.join(OUT_DIR, f"importance_{target}.csv")
    pdp_path = os.path.join(OUT_DIR, f"pdp_{target}.csv")

    # --------------------------------------------------------
    # Skip fully processed targets
    # --------------------------------------------------------
    if os.path.exists(imp_path) and os.path.exists(pdp_path):
        print("Already completed — skipping")
        continue

    # --------------------------------------------------------
    # Load model + features
    # --------------------------------------------------------
    bundle = joblib.load(os.path.join(MODEL_DIR, fname))
    model = bundle["model"]
    input_cols = bundle["input_cols"]

    X = df[input_cols].values

    # Baseline subsample (deterministic)
    X_base = X[
        rng.choice(len(X), size=min(N_BASE, len(X)), replace=False)
    ]

    base_preds = model.predict(X_base)

    # ========================================================
    # 1. PERMUTATION IMPORTANCE (SAVE IMMEDIATELY)
    # ========================================================
    if not os.path.exists(imp_path):
        print("Computing permutation importance")

        importances = []

        for j, feat in enumerate(input_cols):
            Xp = X_base.copy()
            rng.shuffle(Xp[:, j])
            perm_preds = model.predict(Xp)

            importance = np.mean(np.abs(perm_preds - base_preds))
            importances.append((feat, importance))

        imp_df = (
            pd.DataFrame(importances, columns=["feature", "importance"])
            .sort_values("importance", ascending=False)
            .head(TOP_K)
        )

        imp_df.to_csv(imp_path, index=False)
        print("Importance saved")

    else:
        print("Importance already exists")
        imp_df = pd.read_csv(imp_path)

    # ========================================================
    # 2. PARTIAL DEPENDENCE
    # ========================================================
    if os.path.exists(pdp_path):
        pdp_df = pd.read_csv(pdp_path)
        done_features = set(pdp_df["feature"].unique())
        pdp_rows = pdp_df.to_dict("records")
        print(f"Resuming PDPs ({len(done_features)} features done)")
    else:
        pdp_rows = []
        done_features = set()

    for feature in imp_df["feature"]:
        if feature in done_features:
            continue

        print(f"PDP for {feature}")

        j = input_cols.index(feature)
        lo, hi = np.quantile(X_base[:, j], [0.05, 0.95])
        grid = np.linspace(lo, hi, N_PDP)

        for v in grid:
            Xm = X_base.copy()
            Xm[:, j] = v

            pdp_rows.append({
                "feature": feature,
                "feature_value": v,
                "prediction": model.predict(Xm).mean()
            })

        # SAVE AFTER EACH FEATURE (CRASH SAFE)
        pd.DataFrame(pdp_rows).to_csv(pdp_path, index=False)

    print(f"PDPs complete for {target}")

print("\nRandom Forest interpretation finished (restart-safe)")
print(f"Results saved to: {OUT_DIR}")


# III. GLOBAL INTERPRETATION OVERVIEW

## 1. CONFIG & HELPERS

In [ ]:
INTERP_DIR = "results/rf_interpretation"
OUTPUT_SUMMARY = "results/rf_model_interpretation_overview.csv"
OUTPUT_TEXT = "results/rf_model_interpretation_summary.txt"

MIN_EFFECT_SIZE = 0.05
TOP_K = 5

def estimate_slope(pdp_df):
    x = pdp_df["feature_value"].values
    y = pdp_df["prediction"].values
    if len(np.unique(x)) < 2:
        return 0.0
    return np.polyfit(x, y, 1)[0]

## 2. SUMMARIZE TARGETS

In [ ]:
rows = []

targets = sorted(
    set(
        f.replace("importance_", "").replace(".csv", "")
        for f in os.listdir(INTERP_DIR)
        if f.startswith("importance_")
    )
)

print(f"Found {len(targets)} RF targets")

for target in targets:
    print(f"Summarizing {target}")

    imp_path = os.path.join(INTERP_DIR, f"importance_{target}.csv")
    pdp_path = os.path.join(INTERP_DIR, f"pdp_{target}.csv")
    if not os.path.exists(pdp_path):
        continue

    importance_df = pd.read_csv(imp_path).head(TOP_K)
    pdp_df = pd.read_csv(pdp_path)

    for _, row in importance_df.iterrows():
        feature = row["feature"]
        importance = row["importance"]

        feature_pdp = pdp_df[pdp_df["feature"] == feature]
        slope = estimate_slope(feature_pdp)

        if abs(slope) < MIN_EFFECT_SIZE:
            continue  # skip flat features
        direction = "positive" if slope > 0 else "negative"

        rows.append({
            "target": target,
            "feature": feature,
            "importance": importance,
            "effect_direction": direction,
            "effect_slope": slope
        })

## 3. CREATE OVERVIEW TABLE

In [ ]:
overview_df = pd.DataFrame(rows).sort_values(["target", "importance"], ascending=[True, False])
overview_df.to_csv(OUTPUT_SUMMARY, index=False)
print("\nRF interpretation overview created:", OUTPUT_SUMMARY)

# HUMAN-READABLE SUMMARY
summary_rows = []
for target in overview_df["target"].unique():
    subset = overview_df[overview_df["target"] == target]
    pos = subset[subset["effect_direction"] == "positive"]["feature"].tolist()
    neg = subset[subset["effect_direction"] == "negative"]["feature"].tolist()

    statement = []
    if pos:
        statement.append(f"↑ {', '.join(pos[:3])}")
    if neg:
        statement.append(f"↓ {', '.join(neg[:3])}")

    summary_rows.append({"target": target, "summary": " | ".join(statement)})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_TEXT, index=False, header=False)
print(f"Readable summary saved to: {OUTPUT_TEXT}")
